In [3]:
import h5py
import numpy as np
import json
import os
from datetime import datetime

# Definisi elemen target
# versi 35
BASE_ELEMENTS = ["Si", "Al", "Fe", "Ca", "O", "Na", "N", "Ni", "Cr", "Cl", "Mg", "C", "S", "Ar", "Ti", "Mn", "Co"]
REQUIRED_ELEMENTS = [f"{elem}_{i}" for elem in BASE_ELEMENTS for i in [1, 2]]

#versi 9
# BASE_ELEMENTS = ["Al", "Fe", "Ca", "Mg"]
# REQUIRED_ELEMENTS = [f"{elem}_{i}" for elem in BASE_ELEMENTS for i in [1, 2]]

# Konfigurasi
CONFIG = {
    "data": {
        "dataset_path": "dataset2k.h5",
        "element_map_path": "element-map-35.json",
        "train_split": "train",
        "val_split": "validation",
        "max_train_samples": 5000,
        "max_val_samples": 2000,
    },
    "results_dir": "r"
}

def load_element_map(element_map_path):
    """Memuat mapping elemen dari file JSON sebagai dictionary."""
    with open(element_map_path, 'r') as f:
        element_map = json.load(f)
    return element_map

def check_dataset(dataset_path, element_map_path, train_split, val_split, max_train_samples, max_val_samples):
    """Memeriksa proporsi atom dalam dataset."""

    # Muat element map sebagai dictionary
    element_map = load_element_map(element_map_path)
    print(f"{datetime.now()} - Element map dimuat: {list(element_map.keys())}")
    print(f"{datetime.now()} - Number of elements/ions in element_map: {len(element_map)}")

    # Verifikasi panjang nilai one-hot
    if not all(len(one_hot) == 35 for one_hot in element_map.values()):
        print(f"{datetime.now()} - Peringatan: Tidak semua nilai one-hot memiliki panjang 35.")
        return

    # Mapping elemen dasar ke indeks kelas berdasarkan one-hot
    element_to_class = {}
    for elem in BASE_ELEMENTS:
        element_to_class[elem] = []
        for class_name, one_hot in element_map.items():
            if class_name.startswith(elem) and class_name in REQUIRED_ELEMENTS:
                idx = np.argmax(one_hot)  # Indeks kelas berdasarkan one-hot
                element_to_class[elem].append(idx)

    # Mapping kelas ke elemen dasar berdasarkan one-hot
    class_to_element = {}
    for class_name, one_hot in element_map.items():
        idx = np.argmax(one_hot)
        elem = next((e for e in BASE_ELEMENTS if class_name.startswith(e)), "background")
        class_to_element[idx] = elem

    # Inisialisasi dictionary untuk menyimpan distribusi
    split_distributions = {"train": {}, "validation": {}}

# Hapus bagian element_counts dan element_proportions
    with h5py.File(dataset_path, 'r') as h5_file:
        for split, max_samples in [(train_split, max_train_samples), (val_split, max_val_samples)]:
            if split not in h5_file:
                print(f"{datetime.now()} - Split '{split}' tidak ditemukan dalam {dataset_path}")
                continue

            group = h5_file[split]
            spectra = group['spectra'][:max_samples]
            labels = group['labels'][:max_samples]
            print(f"{datetime.now()} - Memproses split: {split}, Spectra shape: {spectra.shape}, Labels shape: {labels.shape}")

            # Hitung distribusi kelas di semua posisi
            all_labels = labels.reshape(-1, 35)  # Flattening ke semua posisi
            class_counts = np.sum(all_labels, axis=0)  # Jumlah 1 di setiap kelas
            total_positions = all_labels.shape[0]  # Total posisi (sampel × 4096)

            # Distribusi per kelas
            split_distributions[split] = {}
            for idx, count in enumerate(class_counts):
                class_name = next((name for name, one_hot in element_map.items() if np.argmax(one_hot) == idx), f"Class_{idx}")
                proportion = count / total_positions * 100
                split_distributions[split][class_name] = {"count": int(count), "proportion (%)": round(proportion, 2)}

            # Cetak hasil
            print(f"\n{datetime.now()} - Distribusi Kelas untuk {split} (Total Posisi: {total_positions:,}):")
            for class_name, stats in split_distributions[split].items():
                print(f"  {class_name}: {stats['count']:,} positions ({stats['proportion (%)']:.2f}%)")

            # Simpan hasil ke file
            os.makedirs(CONFIG["results_dir"], exist_ok=True)
            report_path = os.path.join(CONFIG["results_dir"], f"dataset_distribution_{split}.txt")
            with open(report_path, "w") as f:
                f.write(f"Dataset Distribution Report - {split} (Generated: {datetime.now()})\n")
                f.write(f"Total Positions: {total_positions:,}\n\n")
                f.write("Class Distribution:\n")
                for class_name, stats in split_distributions[split].items():
                    f.write(f"  {class_name}: {stats['count']:,} positions ({stats['proportion (%)']:.2f}%)\n")
            print(f"{datetime.now()} - Laporan distribusi disimpan ke {report_path}")

if __name__ == "__main__":
    check_dataset(
        CONFIG["data"]["dataset_path"],
        CONFIG["data"]["element_map_path"],
        CONFIG["data"]["train_split"],
        CONFIG["data"]["val_split"],
        CONFIG["data"]["max_train_samples"],
        CONFIG["data"]["max_val_samples"]
    )

2025-06-15 00:45:06.489240 - Element map dimuat: ['background', 'Si_1', 'Si_2', 'Al_1', 'Al_2', 'Fe_1', 'Fe_2', 'Ca_1', 'Ca_2', 'O_1', 'O_2', 'Na_1', 'Na_2', 'N_1', 'N_2', 'Ni_1', 'Ni_2', 'Cr_1', 'Cr_2', 'Cl_1', 'Cl_2', 'Mg_1', 'Mg_2', 'C_1', 'C_2', 'S_1', 'S_2', 'Ar_1', 'Ar_2', 'Ti_1', 'Ti_2', 'Mn_1', 'Mn_2', 'Co_1', 'Co_2']
2025-06-15 00:45:06.489460 - Number of elements/ions in element_map: 35
2025-06-15 00:45:08.932840 - Memproses split: train, Spectra shape: (1389, 4096), Labels shape: (1389, 4096, 35)

2025-06-15 00:45:09.265959 - Distribusi Kelas untuk train (Total Posisi: 5,689,344):
  background: 2,202,831 positions (38.72%)
  Si_1: 1,582 positions (0.03%)
  Si_2: 134,448 positions (2.36%)
  Al_1: 1,394 positions (0.02%)
  Al_2: 155,905 positions (2.74%)
  Fe_1: 37,412 positions (0.66%)
  Fe_2: 593,439 positions (10.43%)
  Ca_1: 6,150 positions (0.11%)
  Ca_2: 46,142 positions (0.81%)
  O_1: 452 positions (0.01%)
  O_2: 32,774 positions (0.58%)
  Na_1: 2,233 positions (0.04%)


In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import json
import random
import textwrap

# --- 1. KONFIGURASI PATH UNTUK PENGECEKAN ---
# Path ini seharusnya sudah benar jika Anda menjalankan sel sebelumnya
H5_FILE_TO_CHECK = FINAL_OUTPUT_PATH 
ELEMENT_MAP_PATH = "/Users/namaanda/path/to/element-map-35.json" # Ganti dengan path Anda


# --- 2. FUNGSI UNTUK PLOTTING ---
def plot_sample(h5_file_path: str, element_map_path: str, split: str, sample_idx: int = None):
    """
    Memuat satu sampel dari dataset HDF5 dan mem-plot spektrum beserta labelnya.
    """
    print(f"Membuka file dataset: {h5_file_path}")
    
    try:
        with h5py.File(h5_file_path, 'r') as f:
            if split not in f:
                raise KeyError(f"Split '{split}' tidak ditemukan. Pilihan: {list(f.keys())}")
            
            wavelengths = f['wavelengths'][:]
            num_samples = f[split]['spectra'].shape[0]

            if sample_idx is None:
                sample_idx = random.randint(0, num_samples - 1)
                print(f"Memilih sampel acak dari '{split}' set: Index {sample_idx}")
            else:
                print(f"Memilih sampel spesifik dari '{split}' set: Index {sample_idx}")

            spectrum_data = f[split]['spectra'][sample_idx, :]
            labels_data = f[split]['labels'][sample_idx, :, :]
            metadata_str = f[split]['atom_percentages'][sample_idx].decode('utf-8')
            metadata = json.loads(metadata_str)

        with open(element_map_path, 'r') as f:
            element_map = json.load(f)
        idx_to_name = {np.argmax(one_hot): name for name, one_hot in element_map.items()}
        
    except Exception as e:
        print(f"Error saat memuat data: {e}")
        return

    print("\n" + "="*50)
    print(f"METADATA UNTUK SAMPEL INDEX {sample_idx}")
    print(f"  Suhu              : {metadata.get('temperature', 'N/A'):.2f} K")
    print(f"  Kepadatan Elektron : {metadata.get('electron_density', 'N/A'):.2e} cm^-3")
    element_percentages = {k: v for k, v in metadata.items() if k not in ['temperature', 'electron_density', 'delta_E_max', 'sample_id']}
    element_list_str = ", ".join([f"{k} ({v:.2f}%)" for k, v in sorted(element_percentages.items(), key=lambda item: item[1], reverse=True)])
    print("  Komposisi Elemen (%) :")
    print(textwrap.fill(element_list_str, width=70, initial_indent='    ', subsequent_indent='    '))
    print("="*50 + "\n")
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle(f'Visualisasi Sampel #{sample_idx} dari Set "{split.upper()}"', fontsize=16)

    ax1.plot(wavelengths, spectrum_data, linewidth=1, color='dodgerblue')
    ax1.set_title("Spektrum Emisi Gabungan")
    ax1.set_ylabel("Intensitas Ternormalisasi")
    ax1.grid(True, linestyle=':', alpha=0.6)
    ax1.set_ylim(bottom=0)

    label_image = labels_data.T
    num_classes = label_image.shape[0]
    im = ax2.imshow(label_image, aspect='auto', cmap='viridis', interpolation='nearest',
                    extent=[wavelengths[0], wavelengths[-1], num_classes - 0.5, -0.5])
    ax2.set_title("Label Elemen Dominan")
    ax2.set_xlabel("Panjang Gelombang (nm)")
    ax2.set_yticks(range(num_classes))
    ax2.set_yticklabels([idx_to_name.get(i, f'Unknown_{i}') for i in range(num_classes)], fontsize=8)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# --- 3. JALANKAN PROSES ---
if os.path.exists(H5_FILE_TO_CHECK) and os.path.exists(ELEMENT_MAP_PATH):
    # Mem-plot sampel acak dari training set
    plot_sample(h5_file_path=H5_FILE_TO_CHECK, element_map_path=ELEMENT_MAP_PATH, split='train')
else:
    print("Peringatan: Path untuk file HDF5 atau element map belum benar. Silakan periksa variabel di atas.")

NameError: name 'FINAL_OUTPUT_PATH' is not defined